# RAG with Minsearch Vector 

Note:
first: We embeded all the documents, we fit the minsearch with the embeded vectore
 (note: we need to be sure that the number of embedded vectors is equal to the number of documents)
 
second: given a query, we embed it, then we do vector search using the minsearch vectors

source: https://www.youtube.com/watch?v=-GBW3g3PVTM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=24

# 1. Import packages

In [1]:
from pathlib import Path
from rich import print
import os
import sys
import numpy as np
from minsearch import VectorSearch


parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer
from langchain_core.documents import Document

# 2. Fitting the minsearch 

In [3]:

# 1. load the documents
print('---------------------------------------------------------------')
print('Step 1: load the data')
documents = load_faq_data()
print(f'The number of documents is: {len(documents)}')
# print(documents[0])


print('---------------------------------------------------------------')
print('Step 2: Get the useful data only from each documant')
# extract only the question and answer from each documents
texts = list()

for doc in documents:
    text_per_document = doc['question'] + " " + doc['answer']
    texts.append(text_per_document)

print(f'The number of text documents is: {len(texts)}')
# print(texts[0])

print('---------------------------------------------------------------')
print('Step 3: chunck the data')
# 3.1. Convert your list of text strings into a list of LangChain Document objects
# This keeps your 1,350 documents separate and un-merged.
langchain_docs = [Document(page_content=text) for text in texts]


# Note::: I changed the chunck_size to 5000 so that I get the same numbers of vectors and documents to overcome an error
# 3.2. Setup the Recursive Text Splitter
# It chunks your text smartly, keeping paragraphs/sentences together, and creates overlaps!
# NOTE::: the number of vectors need to be exactly as the number of documents if I use minsearch
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,    # A high number ensures a single Q&A text never gets split up
    # chunk_size=500, # for improvement
    chunk_overlap=0,    # Zero overlap ensures no repeated data, just like your old loop
    # chunk_overlap=50, # for improvement
    # length_function=lambda text: len(tokenizer.encode(text)) # Counting tokens, # for improvement and by default length_function=len
    # separators=[r"Question \d+:", "\n\n", "\n", " "],# for improvement. note custome seperator= separators=["\n===\n", "\n\n", "\n", " "] but since I have only question and answer, I set it differently
    # is_separator_regex=True # for improvement 
)

# split the docs (chuncks)
texts_chuncks = text_splitter.split_documents(langchain_docs) # Note: we do not use  text_splitter.split_text(" ".join(texts))  since we have a list of document, each document will be splitted into chuncks, then langchain merges all the chunck

print(f"The number of chunks documents is: {len(texts_chuncks)}")
# print(texts_chuncks[0])


print('---------------------------------------------------------------')
print('Step 4: embed the data')
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-V2", # same model as in datatalk
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True} # this is for normalizing (sumation =1 of a vector)
)
vectors = embeddings.embed_documents([doc.page_content for doc in texts_chuncks ])

# cast the list to array
array_vectors = np.array(vectors)
print(f'the number of vectors is: {len(array_vectors)}')


print('---------------------------------------------------------------')
print('Step 5: create an index and fit the data')
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(array_vectors, documents)

---------------------------------------------------------------

Step 1: load the data

The number of documents is: 1350

---------------------------------------------------------------

Step 2: Get the useful data only from each documant

...

---------------------------------------------------------------

Step 4: embed the data

the number of vectors is: 1350

---------------------------------------------------------------

Step 5: create an index and fit the data

In [4]:
array_vectors.shape

(1350, 384)

# RAG with modified search that takes the minsearch

In [5]:
# RAGBase

class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.embed_query(query)
        query_vector = np.array(query_vector)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [7]:

# # test to check if ollama is working correctly
# from langchain_ollama import ChatOllama

# llm = ChatOllama(
#     model="gemma3:4b",
#     base_url="http://localhost:11434"
# )

# llm.invoke("Hello")

In [7]:
query = 'I just found out about the program, can I still sign up?' 
assistant = RAGVector(index=vindex, embedder=embeddings)
# for checking
# search_results = assistant.search(query)

# build_context_results = assistant.build_context(search_results=search_results)
# # print(build_context_results)
# chain = assistant.get_chain()
# print(type(chain))

# prompt_value = assistant.prompt.invoke({
#     "context": build_context_results,
#     "question": query,
#     "chat_history": []
# })

# print(prompt_value)


# chain.invoke({
#                         "context": build_context_results,
#                         "question":  query,
#                         "chat_history": assistant.chat_history
#                 })

response = assistant.rag( query=query)
response

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [8]:
assistant.rag('the program has already begun, can I still sign up?')

"I don't know."

In [9]:
assistant.rag('whats queen gambit?')

"I don't know."